In [5]:
import pandas as pd
from groq import Groq
import os
import sqlite3

secound phase

In [8]:
# Load the datasets
heart_rate_df = pd.read_csv('../../data/processed/daily_heart_rate_summary.csv')
oxygen_saturation_df = pd.read_csv('../../data/processed/daily_oxygen_saturation_summary.csv')
steps_df = pd.read_csv('../../data/processed/daily_steps_summary.csv')
posture_df = pd.read_csv('../../data/processed/posture_analysis.csv')
voice_df = pd.read_csv('../../data/processed/voice_analysis.csv')

# Convert date columns to datetime objects
heart_rate_df['date'] = pd.to_datetime(heart_rate_df['date'])
oxygen_saturation_df['date'] = pd.to_datetime(oxygen_saturation_df['date'])
steps_df['date'] = pd.to_datetime(steps_df['date'])
# Convert timestamp to date for posture and voice data, and average if multiple entries per day
posture_df['date'] = pd.to_datetime(posture_df['timestamp']).dt.date
posture_df['date'] = pd.to_datetime(posture_df['date'])
voice_df['date'] = pd.to_datetime(voice_df['timestamp']).dt.date
voice_df['date'] = pd.to_datetime(voice_df['date'])

# Group by date and average the values for posture and voice, as there might be multiple entries per day
posture_daily_df = posture_df.groupby('date').mean(numeric_only=True).reset_index()
voice_daily_df = voice_df.groupby('date').mean(numeric_only=True).reset_index()

# Drop the person_id column as it's not needed for the final merged data
if 'person_id' in posture_daily_df.columns:
    posture_daily_df = posture_daily_df.drop(columns=['person_id'])
if 'person_id' in voice_daily_df.columns:
    voice_daily_df = voice_daily_df.drop(columns=['person_id'])


# Merge the dataframes on the date column
merged_df = pd.merge(heart_rate_df, oxygen_saturation_df, on='date', how='outer')
merged_df = pd.merge(merged_df, steps_df, on='date', how='outer')
merged_df = pd.merge(merged_df, posture_daily_df, on='date', how='outer')
merged_df = pd.merge(merged_df, voice_daily_df, on='date', how='outer')


# Sort the final dataframe by date
merged_df = merged_df.sort_values(by='date')

# Use linear interpolation to fill missing values, then ffill/bfill for any remaining NaNs at the edges
merged_df = merged_df.interpolate(method='linear', limit_direction='both')


# Display the first few rows of the merged and sorted dataframe
print(merged_df.head())

# Display information about the dataframe
print(merged_df.info())

merged_df.to_csv('../../data/processed/merged_health_data.csv', index=False)

        date  avg_heart_rate  max_heart_rate  min_heart_rate  avg_spo2  \
0 2026-02-06           90.50           177.0            62.0     97.27   
1 2026-02-09           72.76           103.0            57.0     97.27   
2 2026-02-10           75.81           114.0            45.0     97.64   
3 2026-02-11           77.41           163.0            50.0     97.65   
4 2026-02-12           77.57           123.0            49.0     97.62   

   max_spo2  min_spo2  daily_step_count  shoulder_asymmetry  slouch_angle  \
0      99.0      96.0            2944.0             1.99292    179.223435   
1      99.0      96.0            2944.0             1.99292    179.223435   
2      99.0      96.0            3751.0             1.99292    179.223435   
3      99.0      92.0            3751.0             1.99292    179.223435   
4      99.0      96.0            3729.0             1.99292    179.223435   

     Jitter   Shimmer        HNR          F1           F2  
0  0.003927  0.082929  15.088124

In [ ]:
# Load the merged dataset
merged_df = pd.read_csv('../../data/processed/merged_health_data.csv')
# Convert date columns to datetime objects
merged_df['date'] = pd.to_datetime(merged_df['date'])


# Configure the Groq API
# For this example, we'll use the key from the environment
# Set GROQ_API_KEY in the environment or a local .env file; never commit it.
client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

# Personal info
personal_info = "جنسیت: مرد - سن: 23 - قد:185 - وزن: 88"

def get_groq_feedback(row):
    """
    Gets feedback from Groq for a given row of health data.
    """
    # Skip rows with missing data
    if row.isnull().any():
        return "اطلاعات ناقص است."

    prompt = f"""
    {personal_info}
    اطلاعات سلامتی امروز من:
    میانگین ضربان قلب: {row['avg_heart_rate']}
    حداکثر ضربان قلب: {row['max_heart_rate']}
    حداقل ضربان قلب: {row['min_heart_rate']}
    میانگین اشباع اکسیژن: {row['avg_spo2']}%
    حداکثر اشباع اکسیژن: {row['max_spo2']}%
    حداقل اشباع اکسیژن: {row['min_spo2']}%
    تعداد قدم‌ها: {row['daily_step_count']}
    زاویه سر به جلو (نشانه قوز): {row['slouch_angle']:.2f} درجه
    عدم تقارن شانه: {row['shoulder_asymmetry']:.2f} پیکسل
    Jitter (لرزش صدا): {row['Jitter'] * 100:.3f}%
    Shimmer (گرفتگی صدا): {row['Shimmer'] * 100:.3f}%
    HNR (نسبت هارمونیک به نویز): {row['HNR']:.2f} dB

    فقط با یک جمله کوتاه و کاربردی (حداکثر ۲۰ کلمه) بگو که وضعیت امروز من چطور بوده و چه کاری برای فردا پیشنهاد می‌کنی. هیچ توضیح اضافه‌ای نده.
    """
    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            model="llama3-8b-8192",
        )
        return chat_completion.choices[0].message.content.strip()
    except Exception as e:
        return f"خطا در ارتباط با Groq: {e}"

# Apply the function to each row of the dataframe
merged_df['feedback'] = merged_df.apply(get_groq_feedback, axis=1)

# Display the dataframe with the new feedback column
print(merged_df[['date', 'feedback']].head())

# Create a connection to the SQLite database
conn = sqlite3.connect('health_data.db')

# Save the dataframe to a table named 'health_feedback'
merged_df.to_sql('health_feedback', conn, if_exists='replace', index=False)

# Close the connection
conn.close()

print("\nData with feedback saved to 'health_data.db' in table 'health_feedback'")